# 🔍 Advanced RAG Techniques with Ollama

A hands-on pipeline that walks you through **six core RAG strategies**, from basic to advanced — all running locally with Ollama.

| # | Technique | What it does |
|---|-----------|-------------|
| 1 | **Dense Embedding** | Semantic similarity search with FAISS |
| 2 | **BM25** | Classic keyword / sparse retrieval |
| 3 | **HyDE** | Generate a *hypothetical* answer, embed it, search with that |
| 4 | **MultiQuery** | Expand one query into many, merge results |
| 5 | **Step-Back** | Ask a broader question first, retrieve on that |
| 6 | **Re-Ranking** | Cross-encoder re-ranks retrieved docs for final scoring |

**Model used everywhere**: `llama3.2` (3B — fast, runs on CPU)  
**Embeddings**: `all-MiniLM-L6-v2` (local, no API key needed)

> �� Run cells top-to-bottom. Each technique builds on the shared corpus and helpers defined early on.


## 📦 Install dependencies

In [ ]:
# Install everything we need (sentence-transformers, faiss, bm25, cross-encoder, ollama client)
!pip install -q sentence-transformers faiss-cpu rank-bm25 ollama
print("✅ Dependencies ready")


## 🦙 Ollama setup

Make sure Ollama is running locally before proceeding.

```bash
# Pull the model once (≈2 GB)
ollama pull llama3.2
```

If you're on **Google Colab**, run the cell below to install + start Ollama automatically.


In [ ]:
import subprocess, time, os, shutil

def is_colab():
    try:
        import google.colab; return True
    except ImportError: return False

if is_colab():
    print("Colab detected — installing Ollama...")

    # zstd is required by the Ollama install script (see issue #14)
    subprocess.run("apt-get install -y zstd", shell=True, check=True)

    # Install Ollama
    result = subprocess.run(
        "curl -fsSL https://ollama.ai/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print("⚠️  Install stderr:", result.stderr[-1000:])
        raise RuntimeError("Ollama install failed")
    print("Install complete.")

    # Locate the binary (install.sh puts it in /usr/local/bin)
    ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"
    if not os.path.exists(ollama_bin):
        raise FileNotFoundError(f"Ollama binary not found at {ollama_bin}. Install stdout: {result.stdout[-500:]}")
    print(f"Ollama binary: {ollama_bin}")

    # Start the server in the background
    subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)  # give it a moment to start

    # Pull the model
    subprocess.run([ollama_bin, "pull", "llama3.2"], check=True)
    print("✅ Ollama ready")
else:
    print("Local run — assuming Ollama is already running with llama3.2 pulled")


## 🔧 Imports & global config

In [ ]:
import re, json, numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi
import ollama

# ── model names ────────────────────────────────────────────────────────
LLM_MODEL   = "llama3.2"          # Ollama model
EMBED_MODEL = "all-MiniLM-L6-v2"  # local embedding model

# ── load models once ───────────────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print("Loading cross-encoder for re-ranking...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Models loaded")


## 📚 Shared document corpus

A small collection of documents about AI / ML concepts.  
All six retrieval techniques will work on this same corpus.


In [ ]:
CORPUS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model. "
    "Instead of relying solely on parametric knowledge, RAG fetches relevant documents at inference time.",

    "BM25 is a bag-of-words ranking function based on the probabilistic model of information retrieval. "
    "It scores documents by term frequency and inverse document frequency, penalising very long documents.",

    "Dense embeddings map text into a high-dimensional vector space so that semantically similar passages "
    "cluster together. Models like sentence-transformers produce these embeddings locally.",

    "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search over dense vectors. "
    "It supports exact and approximate nearest-neighbour algorithms and scales to billions of vectors.",

    "HyDE (Hypothetical Document Embeddings) asks the LLM to write a hypothetical answer to the query, "
    "embeds that answer, and uses the resulting vector for retrieval instead of the raw query.",

    "MultiQuery retrieval expands a single user query into several paraphrased variants using an LLM. "
    "Results from all variants are merged and deduplicated, improving recall.",

    "Step-Back prompting first asks the LLM to rephrase the question at a higher level of abstraction. "
    "Retrieving on this broader question often surfaces background context that narrow queries miss.",

    "Cross-encoder re-rankers score each (query, document) pair jointly, using full attention across both. "
    "They are slower than bi-encoders but significantly more accurate for final ranking.",

    "Hybrid search combines sparse (BM25) and dense (embedding) retrieval scores, typically via reciprocal "
    "rank fusion (RRF) or a weighted sum, to get the best of both worlds.",

    "Chunking strategy matters in RAG: fixed-size chunks, sentence windows, and semantic chunking each "
    "trade precision against recall. Smaller chunks improve precision; larger chunks preserve context.",

    "Llama 3.2 is a compact open-source model from Meta released in 2024. Its 3B variant runs well on "
    "consumer hardware and is suitable for local RAG pipelines.",

    "Knowledge graphs can augment RAG by providing structured entity relationships. GraphRAG walks "
    "the graph to find relevant context, complementing unstructured document retrieval.",
]

# Pre-tokenise for BM25
tokenised_corpus = [doc.lower().split() for doc in CORPUS]
bm25 = BM25Okapi(tokenised_corpus)

# Build FAISS index once
print("Encoding corpus with sentence-transformers...")
corpus_embeddings = embedder.encode(CORPUS, normalize_embeddings=True, show_progress_bar=True)
dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)  # Inner-product = cosine on L2-normalised vectors
faiss_index.add(corpus_embeddings)
print(f"✅ FAISS index built — {faiss_index.ntotal} vectors, dim={dim}")


## 🛠️ Shared helpers

In [ ]:
def llm(prompt: str, system: str = "") -> str:
    """Call the local Ollama LLM and return the response text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = ollama.chat(model=LLM_MODEL, messages=messages)
    return resp["message"]["content"].strip()

def dense_search(query: str, k: int = 3) -> list[tuple[float, str]]:
    """Return top-k (score, doc) pairs from the FAISS index."""
    q_vec = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = faiss_index.search(q_vec, k)
    return [(float(scores[0][i]), CORPUS[idxs[0][i]]) for i in range(k)]

def show_results(results: list[tuple[float, str]], label: str = "Results"):
    print(f"\n{'═'*60}")
    print(f"  {label}")
    print('═'*60)
    for rank, (score, doc) in enumerate(results, 1):
        print(f"\n#{rank}  score={score:.4f}")
        print(f"  {doc[:120]}{'...' if len(doc)>120 else ''}")
    print('═'*60)


---
## 1️⃣ Dense Embedding Retrieval

**How it works**  
1. Encode the query with the same bi-encoder used at index time.  
2. Compute cosine similarity (inner-product on normalised vectors) against all indexed docs.  
3. Return the top-k closest docs.

**When to use**: Semantic queries where exact keywords are absent.


In [ ]:
query = "How does combining retrieval with a language model work?"

results = dense_search(query, k=3)
show_results(results, f"Dense Embedding — '{query}'")


---
## 2️⃣ BM25 (Sparse / Keyword Retrieval)

**How it works**  
BM25 scores documents using term frequency, inverse document frequency, and document length normalisation.  
No neural network — pure statistics, fast, and very strong for exact keyword matches.

**When to use**: Keyword-heavy queries, domain-specific terminology, very short documents.


In [ ]:
query = "BM25 term frequency inverse document frequency ranking"

tokens = query.lower().split()
scores = bm25.get_scores(tokens)

# Pair with docs and sort descending
bm25_results = sorted(
    [(scores[i], CORPUS[i]) for i in range(len(CORPUS))],
    key=lambda x: x[0], reverse=True
)[:3]

show_results(bm25_results, f"BM25 — '{query}'")


---
## 3️⃣ HyDE — Hypothetical Document Embeddings

**How it works**  
1. Give the user's query to the LLM, ask it to *write a hypothetical document* that would answer the query.  
2. Embed the *hypothetical document* (not the original query).  
3. Search the corpus with that embedding.

**Why it helps**: The generated text lives in *document space*, not query space.  
A user's question often uses different vocabulary than the stored documents; HyDE bridges that gap.


In [ ]:
query = "What technique uses a fake generated answer to improve search?"

print(f"Query: {query}\n")
print("Generating hypothetical document...")

hyde_prompt = (
    f"Please write a short passage (2-3 sentences) that would be the ideal answer "
    f"to the following question. Do NOT reference the question itself — just write the passage.\n\n"
    f"Question: {query}"
)
hypothetical_doc = llm(hyde_prompt)
print(f"\n🤖 Hypothetical document:\n{hypothetical_doc}\n")

# Embed the hypothetical document and search
hypo_vec = embedder.encode([hypothetical_doc], normalize_embeddings=True)
scores, idxs = faiss_index.search(hypo_vec, 3)
hyde_results = [(float(scores[0][i]), CORPUS[idxs[0][i]]) for i in range(3)]

show_results(hyde_results, "HyDE results")


---
## 4️⃣ MultiQuery Retrieval

**How it works**  
1. Send the original query to the LLM, ask for N paraphrased variants.  
2. Run dense retrieval for *each* variant.  
3. Merge all result sets, deduplicate, re-sort by best score per document.

**Why it helps**: Different phrasings surface different documents. Recall goes up significantly.


In [ ]:
query = "How do local language models help with private search pipelines?"

print(f"Original query: {query}\n")

# Step 1 — Generate variants
mq_prompt = (
    f"Generate 3 different phrasings of the following search query. "
    f"Each phrasing should emphasise a slightly different aspect. "
    f"Output ONLY the 3 queries, one per line, no numbering.\n\n"
    f"Query: {query}"
)
variants_raw = llm(mq_prompt)
variants = [v.strip() for v in variants_raw.strip().splitlines() if v.strip()][:3]
all_queries = [query] + variants

print("📋 Query variants:")
for q in all_queries:
    print(f"  • {q}")

# Step 2 — Retrieve for each variant
seen: dict[str, float] = {}   # doc -> best score
for q in all_queries:
    for score, doc in dense_search(q, k=3):
        if doc not in seen or score > seen[doc]:
            seen[doc] = score

# Step 3 — Sort merged results
mq_results = sorted(seen.items(), key=lambda x: x[1], reverse=True)[:5]
mq_results = [(score, doc) for doc, score in mq_results]

show_results(mq_results, "MultiQuery merged results")


---
## 5️⃣ Step-Back Prompting

**How it works**  
1. Ask the LLM to rephrase the specific query as a *more general / abstract* question.  
2. Retrieve on that broader question.  
3. Optionally retrieve on the original query too and merge.

**Why it helps**: Narrow questions sometimes miss important background documents.  
Stepping back one level of abstraction pulls in foundational context.


In [ ]:
query = "How does FAISS store vectors for fast lookup at query time?"

print(f"Original query: {query}\n")

# Step 1 — Generate step-back question
sb_prompt = (
    f"Given the following specific question, produce a single more general, "
    f"abstract version that captures the broader concept being asked about. "
    f"Output ONLY the generalised question, nothing else.\n\n"
    f"Specific question: {query}"
)
step_back_query = llm(sb_prompt)
print(f"🔼 Step-back query: {step_back_query}\n")

# Step 2 — Retrieve on step-back query
sb_results = dense_search(step_back_query, k=3)

# Step 3 — Also retrieve on original, merge
original_results = dense_search(query, k=3)
seen = {}
for score, doc in sb_results + original_results:
    if doc not in seen or score > seen[doc]:
        seen[doc] = score

merged = sorted([(s, d) for d, s in seen.items()], reverse=True)[:5]
show_results(merged, "Step-Back + original merged results")


---
## 6️⃣ Re-Ranking with a Cross-Encoder

**How it works**  
1. Use any retrieval strategy (embedding, BM25, …) to get a **candidate set** (e.g. top-10).  
2. Feed every `(query, candidate_doc)` pair to a **cross-encoder** model that scores them jointly.  
3. Re-sort by the cross-encoder scores.

**Why it helps**: Bi-encoders are fast but approximate (query and doc encoded separately).  
Cross-encoders see both at once — much more accurate, but slower.  
The two-stage approach gets speed *and* accuracy.


In [ ]:
query = "What methods improve the accuracy of retrieved documents in RAG?"

print(f"Query: {query}\n")

# Stage 1 — retrieve a wider candidate pool (top-8 from dense)
q_vec = embedder.encode([query], normalize_embeddings=True)
scores, idxs = faiss_index.search(q_vec, 8)
candidates = [CORPUS[idxs[0][i]] for i in range(8)]

print(f"Stage 1 — {len(candidates)} candidates retrieved by dense embedding")

# Stage 2 — cross-encoder re-rank
pairs = [(query, doc) for doc in candidates]
ce_scores = cross_encoder.predict(pairs)

reranked = sorted(zip(ce_scores, candidates), reverse=True)[:4]
show_results(reranked, "Re-ranked results (cross-encoder)")


---
## 🔗 Full Pipeline — All techniques chained

Here we wire everything into a single `advanced_rag()` function:

```
1. MultiQuery  →  expand query
2. Step-Back   →  add broader context
3. BM25        →  catch keyword hits
4. Dense       →  catch semantic hits
5. Merge all candidates
6. Re-rank     →  final cross-encoder pass
7. Generate    →  LLM answer from top docs
```


In [ ]:
def advanced_rag(query: str, top_k_final: int = 3, verbose: bool = True) -> str:
    """Full RAG pipeline combining all six techniques."""
    if verbose: print(f"\nQuery: {query}\n{'─'*60}")
    all_candidates: dict[str, float] = {}

    # ── 1. MultiQuery expansion ───────────────────────────────────────
    mq_prompt = (
        f"Generate 2 different phrasings of this search query. "
        f"Output ONLY the queries, one per line, no numbering.\n\nQuery: {query}"
    )
    variants = [v.strip() for v in llm(mq_prompt).splitlines() if v.strip()][:2]
    if verbose: print(f"MultiQuery variants: {variants}")

    # ── 2. Step-Back query ────────────────────────────────────────────
    sb_q = llm(
        f"Give a single more general version of this question. "
        f"Output only the question.\n\nQuestion: {query}"
    )
    if verbose: print(f"Step-back: {sb_q}")

    # ── 3 & 4. Dense retrieval for all queries ────────────────────────
    for q in [query] + variants + [sb_q]:
        for score, doc in dense_search(q, k=4):
            if doc not in all_candidates or score > all_candidates[doc]:
                all_candidates[doc] = score

    # ── 5. BM25 retrieval ─────────────────────────────────────────────
    bm25_scores = bm25.get_scores(query.lower().split())
    for i, score in enumerate(bm25_scores):
        doc = CORPUS[i]
        bm25_norm = float(score) / (max(bm25_scores) + 1e-9)  # normalise to [0,1]
        if doc not in all_candidates or bm25_norm > all_candidates[doc]:
            all_candidates[doc] = bm25_norm

    if verbose: print(f"Total candidates before re-rank: {len(all_candidates)}")

    # ── 6. Cross-encoder re-rank ──────────────────────────────────────
    docs = list(all_candidates.keys())
    pairs = [(query, doc) for doc in docs]
    ce_scores = cross_encoder.predict(pairs)
    reranked = sorted(zip(ce_scores, docs), reverse=True)[:top_k_final]

    context = "\n\n".join(doc for _, doc in reranked)
    if verbose:
        print("\nTop docs after re-rank:")
        for i, (sc, doc) in enumerate(reranked, 1):
            print(f"  #{i} ({sc:.2f})  {doc[:80]}...")

    # ── 7. Generation ─────────────────────────────────────────────────
    answer = llm(
        f"Answer the question using ONLY the provided context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer concisely.",
        system="You are a helpful assistant. Use only the provided context to answer."
    )
    return answer

# ── Run it ─────────────────────────────────────────────────────────────────
answer = advanced_rag("What are the best strategies to improve retrieval quality in a RAG system?")
print(f"\n{'═'*60}")
print("FINAL ANSWER")
print('═'*60)
print(answer)


---
## 🏁 Summary

| Technique | Strength | Weakness |
|-----------|----------|----------|
| **Dense Embedding** | Semantic similarity, handles synonyms | Misses exact keyword matches |
| **BM25** | Fast, exact keywords, no GPU | Blind to semantics |
| **HyDE** | Bridges query-document vocabulary gap | Extra LLM call per query |
| **MultiQuery** | Higher recall, robust to phrasing | Multiple LLM calls + retrieval |
| **Step-Back** | Surfaces background / foundational docs | May over-generalise narrow queries |
| **Re-Ranking** | Most accurate final ordering | Slower (cross-encoder over all candidates) |

**Recommended production stack**: MultiQuery → Dense + BM25 hybrid → Re-rank → Generate  
HyDE and Step-Back are great additions for tricky queries with vocabulary mismatch.
